# OCEAVERA — OBIS Data Cleaning

- **Project:** OCEAVERA — Marine Habitat Intelligence
- **Branch:** `pipeline/data-cleaning-obis`
- **Review date:** 11 October 2026


## Contributors

<table style="width:100%; border-collapse:separate; border-spacing:1rem .2rem; margin:.45rem 0 1.25rem;">
<thead>
<tr>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">Registration Number</th>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">Member Name</th>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">GitHub Account</th>
</tr>
</thead>
<tbody>
<tr>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">IT24100788</td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">Sanuda Abeysinghe</td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;"><code>sanudaabey</code></td>
</tr>
</tbody>
</table>

This is a branch-level implementation record, not an individual contribution statement or assessed reflection.


## 1. Purpose and scope

This branch creates a generic, reproducible cleaned OBIS occurrence table for later biological-data processing. It keeps records across multiple taxa and preserves identifiers, source context, coordinates, dates and quality fields for later review.

### Work implemented

1. `stage_obis_csv.py` validates the selected raw CSV's structure, records its dimensions and SHA-256 in a manifest, and does not copy or edit the raw file.
2. `clean_obis_occurrences_csv.py` checks the raw source against that manifest, measures column completeness, applies the column and row rules below, and writes the current cleaned CSV and JSON diagnostics.
3. The outputs and decisions are documented in this notebook and the linked cleaning record.

### Deliberate boundaries

This is generic column and row cleaning only. It does not select species, reconcile taxonomy, inspect environmental rasters, create an integrated dataset, construct presence/background labels, engineer features or train models. It does not treat missing occurrences as confirmed absences. Under D-049, `data/interim/obis/` has one active stage directory, `cleaning/`, which replaces the former source-validation directory.


## 2. Input data and provenance

The source is the saved OBIS Area 230 CSV. The source validator records the fingerprint below; the cleaner checks it before and after reading the raw file.

| Item | Recorded value |
| --- | --- |
| Raw source | `data/raw/obis/csv/area-230-20261008T055217394846Z.csv` |
| Source records and columns | 23,934 × 230 |
| Source size | 26,892,051 bytes |
| Raw SHA-256 | `16ec5ff96e57dc1e58baa33912d16fe7e9098c78759e750608611bf9a2dc0c43` |
| Manifest | `data/interim/obis/cleaning/source_validation_manifest.json` |
| Current output directory | `data/interim/obis/cleaning/` |

The raw file remains the source of truth. The manifest records a fingerprint; it is not a second staged CSV.


## 3. Column-cleaning method

Completeness is calculated over all 23,934 raw records. A value counts as usable when it is not blank or whitespace and is not a confirmed missing sentinel for that specific field. Sentinel matching is case-insensitive and used only for completeness diagnostics; source values are not imputed or rewritten.

The configured minimum is **10% usable values**.

An earlier exploratory 90% screen retained 22 columns; it was not adopted because it discarded fields before project-specific review. The 10% minimum is a pragmatic initial completeness policy, not a universal scientific threshold or a final predictor list. A non-protected column is retained when it meets or exceeds that threshold. Protected fields remain even below the threshold.

### Protected fields

All 37 requested fields were present in the source and are retained:

- **Record and provenance:** `id`, `occurrenceID`, `dataset_id`, `datasetName`, `license`, `eventID`
- **Taxonomic identity:** `scientificName`, `originalScientificName`, `species`, `speciesid`, `AphiaID`, `aphiaID`, `scientificNameID`, `taxonRank`
- **Occurrence:** `occurrenceStatus`, `absence`, `basisOfRecord`, `occurrenceRemarks`
- **Geography:** `decimalLatitude`, `decimalLongitude`, `coordinateUncertaintyInMeters`, `marine`, `locality`
- **Dates:** `eventDate`, `date_year`, `date_start`, `date_end`, `date_mid`
- **Environmental attributes:** `bathymetry`, `shoredistance`, `sst`, `sss`
- **Quality flags:** `flags`, `dropped`
- **Habitat and sampling metadata:** `habitat`, `samplingProtocol`, `samplingEffort`

### Field-specific missing sentinels

- The text `NA` is treated as missing in `lifeStage`, `originalScientificName` and `specificEpithet`.
- The text `null` is treated as missing in `day`, `islandGroup`, `modified`, `month`, `verbatimElevation` and `year`.
- These rules do not turn arbitrary text such as `NA` into a missing value in unrelated fields.

### Column result

| Result | Count |
| --- | ---: |
| Source columns | 230 |
| Columns meeting the 10% threshold | 108 |
| Protected columns below the threshold but retained | 4 |
| Total retained | 112 |
| Removed below threshold | 118 |
| Protected columns retained | 37 of 37 |

The four protected fields below the threshold are `eventID`, `habitat`, `samplingEffort` and `samplingProtocol`. `lifeStage` had 1,547 usable records (6.46%) after 2,806 `NA` sentinel values were counted as missing, so it was removed. The JSON column report gives the completeness and retention reason for every original field.


## 4. Row-cleaning method

The rules require meaningful information for a general biological occurrence without making species-level taxonomy or dates mandatory. The cleaner applies the first matching exclusion rule in this order:

1. Exclude records explicitly marked `dropped=true` by OBIS.
2. Exclude records whose latitude or longitude is absent, non-numeric, non-finite or outside the legal coordinate range.
3. Exclude records with no usable taxonomic name or identifier at any available rank. The eligibility set includes `taxonID`, scientific names, species identifiers and higher-rank identifiers.
4. Exclude only confirmed exact duplicates across **all original source columns**, checked before column reduction.

There is no blanket row-missingness cutoff. Such a score would count optional metadata as if it were required and could duplicate the explicit eligibility checks. A record is not removed solely for missing `eventDate`, species-level fields, `taxonRank`, coordinate uncertainty or sampling metadata. The cleaner preserves `absence`, `marine` and quality flags for later interpretation.


## 5. Cleaning outcomes

The exclusion counts are mutually exclusive because the cleaner assigns each excluded row to its first matching rule.

| Rule | Rows excluded |
| --- | ---: |
| OBIS `dropped=true` | 563 |
| Invalid or unusable coordinates | 0 |
| No usable taxonomic identification | 0 |
| Confirmed exact duplicate across original columns | 0 |
| **Total excluded** | **563** |
| **Final cleaned records** | **23,371** |

The source had no exact duplicate rows. Repeated identifiers and species/location/date combinations were reported, not treated as proof of duplicate observations.


## 6. Missing optional information and retained quality data

These counts describe the 23,371 cleaned rows. They are reported for later review and did not trigger row removal.

| Optional information | Missing records | Missing |
| --- | ---: | ---: |
| All observation-date fields missing | 5,043 | 21.58% |
| Both `species` and `speciesid` missing | 5,952 | 25.47% |
| `coordinateUncertaintyInMeters` missing | 12,407 | 53.09% |
| `taxonRank` missing | 13,033 | 55.77% |
| `habitat` missing | 23,102 | 98.85% |
| `samplingEffort` missing | 22,407 | 95.88% |
| `samplingProtocol` missing | 22,534 | 96.42% |

The output keeps 3,188 distinct non-empty species names. It also retains 115 `marine=false` records and 52 records with blank `marine`; 44 records have `absence=true`. Neither marine status nor the absence field was used as a blanket exclusion rule.

OBIS quality-flag counts include `ON_LAND` (8,000), `NO_DEPTH` (16,891), `NO_ACCEPTED_NAME` (490), `DEPTH_EXCEEDS_BATH` (584) and `NO_MATCH` (0). Flag counts can overlap; these flags were preserved and were not, by themselves, row-exclusion rules.


## 7. Duplicate diagnostics

| Diagnostic | Source | Cleaned output | Treatment |
| --- | ---: | ---: | --- |
| Exact full-row duplicate groups | 0 | 0 | No rows removed for this reason |
| Repeated non-empty `occurrenceID` groups | 58 groups / 116 extra records | 58 groups / 116 extra records | Preserved; identifier repetition alone does not prove duplicate identity |
| Repeated species/coordinate/date groups | 501 groups / 2,146 extra records | 496 groups / 2,136 extra records | Preserved; repeated sampling combinations can be distinct source records |

The complete diagnostics are in `cleaning_summary.json`. Exact duplicate detection uses the original source columns, so dropping low-completeness metadata cannot create a false duplicate match.


## 8. Branch outputs and current interim layout

| Artefact | Purpose |
| --- | --- |
| `data/interim/obis/cleaning/occurrences_cleaned.csv` | Primary cleaned dataset: 23,371 rows × 112 columns |
| `data/interim/obis/cleaning/column_completeness.json` | Per-column populated/missing counts, sentinel counts, completeness percentage and retention reason |
| `data/interim/obis/cleaning/cleaning_summary.json` | Source fingerprint, rules, exclusion counts, optional missingness, duplicate diagnostics, preserved statuses and output hashes |
| `data/interim/obis/cleaning/source_validation_manifest.json` | Raw source path, dimensions, byte count and SHA-256 |

D-049 specifies one current dataset per interim data family. OBIS therefore has only `cleaning/` directly under `data/interim/obis/`, with one CSV. Bio-ORACLE's current `source_validation/` collection is unaffected; a future validated Bio-ORACLE cleaning stage must replace that directory.


## 9. Reproduction

Run from the repository root with the project Python environment. These commands validate the raw CSV and then regenerate the current OBIS cleaned output and diagnostics:

```text
python src/data_preparation/stage_obis_csv.py --obis-csv data/raw/obis/csv/area-230-20261008T055217394846Z.csv
python src/data_preparation/clean_obis_occurrences_csv.py
```

Both scripts use the Python standard library. They do not modify the raw CSV or access Bio-ORACLE.


### Verification 1 — Source and output inventory

This cell reads the saved source-validation manifest and raw CSV, recomputes the raw source hash and dimensions, and confirms the one-directory/one-CSV OBIS interim layout. It does not write files.


In [13]:
from pathlib import Path
import csv
import hashlib
import json
import sys

ROOT = Path.cwd()
while not (ROOT / "src/data_preparation/clean_obis_occurrences_csv.py").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Could not locate the OCEAVERA repository root.")
    ROOT = ROOT.parent

OBIS_DIR = ROOT / "data/interim/obis"
OUTPUT_DIR = OBIS_DIR / "cleaning"
MANIFEST = OUTPUT_DIR / "source_validation_manifest.json"
CLEANED = OUTPUT_DIR / "occurrences_cleaned.csv"
COLUMN_REPORT = OUTPUT_DIR / "column_completeness.json"
RUN_SUMMARY = OUTPUT_DIR / "cleaning_summary.json"
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
RAW = ROOT / manifest["source"]["path"]

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

with RAW.open("r", encoding="utf-8-sig", newline="") as stream:
    reader = csv.reader(stream, strict=True)
    raw_header = next(reader)
    raw_rows = sum(1 for _ in reader)

interim_directories = [path for path in OBIS_DIR.iterdir() if path.is_dir()]
interim_csvs = list(OBIS_DIR.rglob("*.csv"))
source_info = manifest["source"]

assert manifest["format"] == "oceavera-obis-interim-source-validation-v2"
assert sha256(RAW) == source_info["sha256"]
assert RAW.stat().st_size == source_info["bytes"]
assert (raw_rows, len(raw_header)) == (source_info["row_count"], source_info["column_count"])
assert interim_directories == [OUTPUT_DIR]
assert interim_csvs == [CLEANED]

print(f"Python: {sys.version.split()[0]}")
print(f"Raw source: {RAW.relative_to(ROOT).as_posix()}")
print(f"Source shape and size: {raw_rows:,} rows x {len(raw_header)} columns; {RAW.stat().st_size:,} bytes")
print(f"Raw SHA-256 matches manifest: {source_info['sha256']}")
print("OBIS interim layout: one cleaning/ directory; one CSV")


Python: 3.14.8
Raw source: data/raw/obis/csv/area-230-20261008T055217394846Z.csv
Source shape and size: 23,934 rows x 230 columns; 26,892,051 bytes
Raw SHA-256 matches manifest: 16ec5ff96e57dc1e58baa33912d16fe7e9098c78759e750608611bf9a2dc0c43
OBIS interim layout: one cleaning/ directory; one CSV


### Verification 2 — Column-retention results

This cell reconciles the 230-column completeness report with the configured threshold and protected-field rules. It prints the least-complete removed fields and those nearest to the cutoff; the JSON report remains the full per-column record.


In [14]:
column_report = json.loads(COLUMN_REPORT.read_text(encoding="utf-8"))
run_summary = json.loads(RUN_SUMMARY.read_text(encoding="utf-8"))
column_rows = column_report["columns"]
source_rows = column_report["source_rows"]
threshold = column_report["minimum_completeness"]
retained = [row for row in column_rows if row["retained"]]
removed = [row for row in column_rows if not row["retained"]]
protected = [row for row in column_rows if row["protected"]]
threshold_columns = [row for row in column_rows if row["usable_records"] / source_rows >= threshold]
protected_below = [row["column"] for row in protected if row["retention_reason"] == "protected below threshold"]
life_stage = next(row for row in column_rows if row["column"] == "lifeStage")
sentinel_counts = [row for row in column_rows if row["missing_sentinel_records"]]

assert source_rows == 23934 and len(column_rows) == 230
assert threshold == 0.10
assert len(retained) == 112 and len(removed) == 118
assert len(threshold_columns) == 108 and len(protected) == 37
assert len(protected_below) == 4
assert set(protected_below) == {"eventID", "habitat", "samplingEffort", "samplingProtocol"}
assert all(row["retained"] for row in protected)
assert life_stage["usable_records"] == 1547
assert life_stage["missing_sentinel_records"] == 2806
assert not life_stage["retained"]
assert run_summary["columns"]["protected_missing_from_source"] == []

print(f"Source columns: {len(column_rows)}; threshold: {threshold:.0%}")
print(f"Retained: {len(retained)}; removed: {len(removed)}; protected retained: {len(protected)}")
print(f"Meet threshold: {len(threshold_columns)}; protected below threshold: {', '.join(sorted(protected_below))}")
print(f"lifeStage: {life_stage['usable_records']:,}/{source_rows:,} usable ({life_stage['percent_populated']:.2f}%); removed")
print("Missing sentinels counted by field:")
for row in sentinel_counts:
    print(f"  {row['column']}: {row['missing_sentinel_records']:,}")
print("Least-complete removed columns:")
for row in sorted(removed, key=lambda item: (item["percent_populated"], item["column"]))[:10]:
    print(f"  {row['column']}: {row['percent_populated']:.2f}%")
print("Removed columns nearest the 10% cutoff:")
for row in sorted(removed, key=lambda item: (-item["percent_populated"], item["column"]))[:8]:
    print(f"  {row['column']}: {row['percent_populated']:.2f}%")
print("All per-column counts and decisions: data/interim/obis/cleaning/column_completeness.json")


Source columns: 230; threshold: 10%
Retained: 112; removed: 118; protected retained: 37
Meet threshold: 108; protected below threshold: eventID, habitat, samplingEffort, samplingProtocol
lifeStage: 1,547/23,934 usable (6.46%); removed
Missing sentinels counted by field:
  day: 2
  islandGroup: 2
  lifeStage: 2,806
  modified: 10
  month: 2
  originalScientificName: 47
  specificEpithet: 3,291
  verbatimElevation: 2
  year: 2
Least-complete removed columns:
  domain: 0.00%
  freshwater: 0.00%
  maximumElevationInMeters: 0.00%
  minimumElevationInMeters: 0.00%
  namePublishedInID: 0.00%
  natio: 0.00%
  phylum (division): 0.00%
  redlist_category: 0.00%
  series: 0.00%
  subforma: 0.00%
Removed columns nearest the 10% cutoff:
  infraclass: 9.86%
  infraclassid: 9.86%
  bibliographicCitation: 9.53%
  recordNumber: 8.58%
  continent: 8.52%
  infrakingdom: 8.26%
  infrakingdomid: 8.26%
  accessRights: 7.40%
All per-column counts and decisions: data/interim/obis/cleaning/column_completeness.

### Verification 3 — Row rules, optional missingness and duplicates

This cell reports the cleaner's recorded first-match exclusions and optional-data/duplicate diagnostics. It distinguishes actual row-removal reasons from missing information that was deliberately retained.


In [15]:
summary = run_summary
criteria = summary["criteria"]
rows_summary = summary["rows"]
exclusions = rows_summary["exclusion_counts"]
optional = rows_summary["optional_data_missingness"]
duplicates = summary["duplicate_diagnostics"]
preserved = summary["preserved_values"]

assert exclusions == {
    "OBIS dropped status": 563,
    "invalid coordinates": 0,
    "no usable taxonomic identification": 0,
    "confirmed exact duplicate": 0,
    "total": 563,
}
assert rows_summary["source_count"] == 23934
assert rows_summary["retained_count"] == 23371
assert optional["all_observation_date_fields_missing"]["missing_records"] == 5043
assert optional["both_species_and_speciesid_missing"]["missing_records"] == 5952
assert optional["fields"]["coordinateUncertaintyInMeters"]["missing_records"] == 12407
assert optional["fields"]["taxonRank"]["missing_records"] == 13033
assert duplicates["exact_source_rows_groups_total_extra"] == [0, 0, 0]
assert duplicates["exact_rows_after_column_reduction_groups_total_extra"] == [0, 0, 0]

print("Rows excluded by first applicable rule:")
for reason in ("OBIS dropped status", "invalid coordinates", "no usable taxonomic identification", "confirmed exact duplicate"):
    print(f"  {reason}: {exclusions[reason]:,}")
print(f"Final rows: {rows_summary['retained_count']:,}")
print("Optional information missing in retained rows:")
print(f"  all date fields: {optional['all_observation_date_fields_missing']['missing_records']:,} ({optional['all_observation_date_fields_missing']['percent_missing']:.2f}%)")
print(f"  both species and speciesid: {optional['both_species_and_speciesid_missing']['missing_records']:,} ({optional['both_species_and_speciesid_missing']['percent_missing']:.2f}%)")
for field in ("coordinateUncertaintyInMeters", "taxonRank", "habitat", "samplingEffort", "samplingProtocol"):
    item = optional["fields"][field]
    print(f"  {field}: {item['missing_records']:,} ({item['percent_missing']:.2f}%)")
output_occurrence_ids = duplicates["repeated_occurrenceID_output_groups_total_extra"]
source_combinations = duplicates["repeated_species_coordinate_date_source_groups_total_extra"]
output_combinations = duplicates["repeated_species_coordinate_date_output_groups_total_extra"]
print(f"Repeated occurrenceID groups preserved: {output_occurrence_ids[0]} groups / {output_occurrence_ids[1]} extra records")
print(f"Repeated species-coordinate-date groups: {source_combinations[0]} -> {output_combinations[0]} groups; {source_combinations[1]} -> {output_combinations[1]} extra records")
print(f"Marine statuses preserved: true={preserved['marine']['true']:,}, false={preserved['marine']['false']:,}, blank={preserved['marine']['blank']:,}")
print(f"Absence values preserved: true={preserved['absence']['true']:,}, false={preserved['absence']['false']:,}")
print("Quality-flag counts are in cleaning_summary.json; flags may overlap.")


Rows excluded by first applicable rule:
  OBIS dropped status: 563
  invalid coordinates: 0
  no usable taxonomic identification: 0
  confirmed exact duplicate: 0
Final rows: 23,371
Optional information missing in retained rows:
  all date fields: 5,043 (21.58%)
  both species and speciesid: 5,952 (25.47%)
  coordinateUncertaintyInMeters: 12,407 (53.09%)
  taxonRank: 13,033 (55.77%)
  habitat: 23,102 (98.85%)
  samplingEffort: 22,407 (95.88%)
  samplingProtocol: 22,534 (96.42%)
Repeated occurrenceID groups preserved: 58 groups / 116 extra records
Repeated species-coordinate-date groups: 501 -> 496 groups; 2146 -> 2136 extra records
Marine statuses preserved: true=23,204, false=115, blank=52
Absence values preserved: true=44, false=23,327
Quality-flag counts are in cleaning_summary.json; flags may overlap.


### Verification 4 — Cleaned CSV values and final hashes

The final cell compares every output record with its corresponding raw record, confirms retained-column order and original values, and rechecks coordinate/taxonomic eligibility. It reports aggregate counts and hashes only.


In [16]:
import math

retained_columns = [row["column"] for row in column_rows if row["retained"]]
taxon_fields = tuple(criteria["taxonomy_eligibility_fields"])
missing_sentinels = criteria["missing_value_sentinels"]

def usable(value, field):
    value = value.strip()
    sentinels = {item.casefold() for item in missing_sentinels.get(field, [])}
    return bool(value) and value.casefold() not in sentinels

retained_count = 0
dropped_count = 0
species_names = set()
marine_counts = {"true": 0, "false": 0, "blank": 0}
absence_counts = {"true": 0, "false": 0, "blank": 0}

with CLEANED.open("r", encoding="utf-8-sig", newline="") as output_stream:
    output_reader = csv.DictReader(output_stream, strict=True)
    assert output_reader.fieldnames == retained_columns
    with RAW.open("r", encoding="utf-8-sig", newline="") as source_stream:
        source_reader = csv.DictReader(source_stream, strict=True)
        for source_row in source_reader:
            if source_row.get("dropped", "").strip().casefold() == "true":
                dropped_count += 1
                continue
            output_row = next(output_reader, None)
            assert output_row is not None
            assert all(output_row[name] == source_row[name] for name in retained_columns)
            latitude = float(source_row["decimalLatitude"])
            longitude = float(source_row["decimalLongitude"])
            assert math.isfinite(latitude) and math.isfinite(longitude)
            assert -90 <= latitude <= 90 and -180 <= longitude <= 180
            assert any(usable(source_row.get(name, ""), name) for name in taxon_fields)
            if usable(source_row.get("species", ""), "species"):
                species_names.add(source_row["species"].strip())
            marine = source_row.get("marine", "").strip().casefold() or "blank"
            absence = source_row.get("absence", "").strip().casefold() or "blank"
            if marine in marine_counts:
                marine_counts[marine] += 1
            if absence in absence_counts:
                absence_counts[absence] += 1
            retained_count += 1
    assert next(output_reader, None) is None

assert dropped_count == 563 and retained_count == 23371
assert len(species_names) == 3188
assert marine_counts == {"true": 23204, "false": 115, "blank": 52}
assert absence_counts == {"true": 44, "false": 23327, "blank": 0}
assert sha256(CLEANED) == summary["outputs"]["cleaned_csv"]["sha256"]
assert sha256(COLUMN_REPORT) == summary["outputs"]["column_completeness_json"]["sha256"]

print("Retained values and column order match source: yes")
print("All output coordinates valid and a taxonomic identifier is present: yes")
print(f"Final dataset: {retained_count:,} rows x {len(retained_columns)} columns")
print(f"Rows excluded by OBIS dropped status: {dropped_count:,}; other eligibility exclusions: 0")
print(f"Distinct non-empty species names: {len(species_names):,}")
print(f"Cleaned CSV SHA-256: {sha256(CLEANED)}")
print(f"Column report SHA-256: {sha256(COLUMN_REPORT)}")
print("Notebook made no network requests and wrote no dataset files.")


Retained values and column order match source: yes
All output coordinates valid and a taxonomic identifier is present: yes
Final dataset: 23,371 rows x 112 columns
Rows excluded by OBIS dropped status: 563; other eligibility exclusions: 0
Distinct non-empty species names: 3,188
Cleaned CSV SHA-256: 1e51a18bb6cf096e111dd0f2c810c97e93619501f74574c3b2c6be130b1aec21
Column report SHA-256: 63774d6050980f089e0a979cc3dcf904ccf168a5202251f75e6549baeff6cae5
Notebook made no network requests and wrote no dataset files.


## 10. Verification and limits

The notebook's ordered Code cells use local files in read-only mode. They verify the source hash, size and dimensions; the single OBIS interim stage and CSV; the column decisions; output hashes; row eligibility; and unchanged values for every retained output field. No network access or notebook-generated dataset write is performed.

The notebook checks ran using the available bundled **Python 3.12.14** interpreter and standard library. This records the interpreter actually used for these checks; it does not claim execution in the project's OCEAVERA Conda environment.

Executing these checks confirms the recorded structure and output consistency; it is not scientific validation or human approval. This cleaning work does not establish taxonomic correctness, marine habitat validity, coordinate-to-raster suitability, scientific species eligibility or model readiness. Those questions remain for separately authorised later work.


## 11. References and next-step context

- [OBIS cleaning implementation](../../src/data_preparation/clean_obis_occurrences_csv.py)
- [OBIS source validator](../../src/data_preparation/stage_obis_csv.py)
- [OBIS cleaning decision and evidence record](../../docs/records/2026-10-11-obis-area-230-cleaning.md)
- [One-current-interim-dataset decision D-049](../../docs/records/2026-10-11-decision-interim-one-current-dataset.md)
- [Data storage and provenance](../../docs/data/data-storage-and-provenance.md)
- [Project requirements](../../PROJECT_REQUIREMENTS.md)
- [Repository readiness](../../docs/project/repository-readiness-and-alignment.md)

This notebook explains the branch implementation and observed cleaning outcomes. It does not approve a focal species or start taxonomic, spatial, integration or modelling work.
